# Pruebas de Ajustes Fallidos en Fórmulas Excel

Este notebook realiza pruebas automáticas para verificar si las fórmulas en las columnas Año, Mes, Semestre y LLAVE realmente se escriben como fórmulas en el archivo Excel, después de aplicar los ajustes del pipeline. Se simulan varios intentos fallidos y se reportan los resultados.

In [ ]:
# 1. Importar bibliotecas necesarias
import openpyxl
import pandas as pd
import numpy as np
import os
from openpyxl import Workbook, load_workbook
from openpyxl.utils import get_column_letter
import pytest

# Ruta de prueba (ajustar si es necesario)
RUTA_EXCEL = '../data/output/Resultados Consolidados.xlsx'


In [ ]:
# 2. Definir funciones de ajuste simulando fallos

def intento_ajuste_1(ws):
    # Simula un ajuste que no escribe fórmulas
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        for cell in row:
            if cell.column in [2, 3, 4, 5]:  # Suponiendo columnas Año, Mes, Semestre, LLAVE
                cell.value = 2024  # Valor fijo, no fórmula

def intento_ajuste_2(ws):
    # Simula un ajuste que borra fórmulas
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        for cell in row:
            if cell.column in [2, 3, 4, 5]:
                cell.value = None

def intento_ajuste_3(ws):
    # Simula un ajuste que intenta poner fórmula pero en inglés
    for i, row in enumerate(ws.iter_rows(min_row=2, max_row=ws.max_row), start=2):
        ws.cell(i, 2).value = f'=YEAR(F{i})'
        ws.cell(i, 3).value = f'=PROPER(TEXT(F{i},"mmmm"))'
        ws.cell(i, 4).value = f'=IF(OR(H{i}="Enero",H{i}="Febrero"),G{i}&"-1",G{i}&"-2")'
        ws.cell(i, 5).value = f'=A{i}&"-"&YEAR(F{i})'


In [ ]:
# 3. Intentar ajustes y capturar fallos

def probar_ajustes_y_capturar(ws):
    errores = []
    try:
        intento_ajuste_1(ws)
        raise Exception('Ajuste 1 no escribe fórmulas')
    except Exception as e:
        errores.append(f"Ajuste 1: {e}")
    try:
        intento_ajuste_2(ws)
        raise Exception('Ajuste 2 borra fórmulas')
    except Exception as e:
        errores.append(f"Ajuste 2: {e}")
    try:
        intento_ajuste_3(ws)
        raise Exception('Ajuste 3 usa fórmulas en inglés')
    except Exception as e:
        errores.append(f"Ajuste 3: {e}")
    return errores

# Cargar archivo de prueba y aplicar los intentos
wb = load_workbook(RUTA_EXCEL)
ws = wb.active
fallos = probar_ajustes_y_capturar(ws)
fallos

In [ ]:
# 4. Realizar pruebas unitarias sobre los datos

def test_columnas_con_formula(ws):
    # Verifica que las columnas 2, 3, 4, 5 tengan fórmulas en español
    errores = []
    for i, row in enumerate(ws.iter_rows(min_row=2, max_row=ws.max_row), start=2):
        for col, esperado in zip([2, 3, 4, 5], ['=AÑO', '=NOMPROPIO', '=SI', '=A']):
            val = ws.cell(i, col).value
            if not (isinstance(val, str) and val.replace(' ', '').startswith(esperado)):
                errores.append(f"Fila {i}, Col {col}: No hay fórmula en español, valor: {val}")
    return errores

# Ejecutar la prueba
prueba_errores = test_columnas_con_formula(ws)
prueba_errores

# 5. Mostrar resultados de las pruebas

A continuación se muestran los errores detectados en las pruebas de escritura de fórmulas. Si la lista está vacía, las fórmulas están correctamente escritas en las columnas Año, Mes, Semestre y LLAVE. Si hay mensajes, indica filas/columnas donde no se encontró fórmula en español.